# NB_LIBRO_01 · Figuras de los capítulos 1 y 2 (versión 2)

**Libro:** *AgroVisión: del píxel a la decisión*

Este notebook genera las figuras nuevas de los capítulos 1 y 2 y un archivo con los valores numéricos que el texto necesita.

**Qué usa (solo lectura, no modifica nada de los proyectos):**

- `PALMA_AFRICANA_YOLO/02_DATOS_PROCESADOS/classification_5class_v1` (452 imágenes, 5 clases)
- `AgroUTMACH_PIX/04_VIDEOS_FERIA/PIO_alta_original_45s.mp4`

**Qué produce (en `AGROVISION_LIBRO/`):**

| Archivo | Figura del libro |
|---|---|
| `02_FIGURAS/cap01/Fig_1_1_cadena_pixel_decision.png` | 1.1 |
| `02_FIGURAS/cap01/Fig_1_2_clases_palma.png` | 1.2 |
| `02_FIGURAS/cap02/Fig_2_1_pixel_ampliacion.png` | 2.1 |
| `02_FIGURAS/cap02/Fig_2_2_resolucion_espacial.png` | 2.2 |
| `02_FIGURAS/cap02/Fig_2_3_canales_RGB.png` | 2.3 |
| `02_FIGURAS/cap02/Fig_2_4_canales_HSV.png` | 2.4 |
| `02_FIGURAS/cap02/Fig_2_5_canales_CIELAB.png` | 2.5 |
| `02_FIGURAS/cap02/Fig_2_6_histogramas.png` | 2.6 |
| `02_FIGURAS/cap02/Fig_2_7_color_por_clase.png` | 2.7 |
| `02_FIGURAS/cap02/Fig_2_8_fotogramas_consecutivos.png` | 2.8 |
| `03_VALORES/valores_para_texto.json` | cifras marcadas con ⟦ ⟧ en el manuscrito |
| `03_VALORES/tabla_2_3_color_por_clase.csv` | Tabla 2.3 |

**Cómo ejecutarlo:** menú *Entorno de ejecución → Ejecutar todo*. No necesita GPU. Tarda entre 5 y 10 minutos; casi todo el tiempo se va en leer las 452 imágenes desde Drive.

Todas las figuras se guardan en PNG a 300 dpi, con 16 cm de ancho.

## 1. Configuración

In [ ]:
import os
import json
import math
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

# Carpeta raíz de Drive. Fuera de Colab puede indicarse con la variable AGROVISION_BASE.
BASE = Path(os.environ.get('AGROVISION_BASE', '/content/drive/MyDrive'))

DIR_PALMA = BASE / 'PALMA_AFRICANA_YOLO' / '02_DATOS_PROCESADOS' / 'classification_5class_v1'
VIDEO_PIO = BASE / 'AgroUTMACH_PIX' / '04_VIDEOS_FERIA' / 'PIO_alta_original_45s.mp4'

DIR_LIBRO = BASE / 'AGROVISION_LIBRO'
DIR_FIG1 = DIR_LIBRO / '02_FIGURAS' / 'cap01'
DIR_FIG2 = DIR_LIBRO / '02_FIGURAS' / 'cap02'
DIR_VAL = DIR_LIBRO / '03_VALORES'
for d in (DIR_FIG1, DIR_FIG2, DIR_VAL):
    d.mkdir(parents=True, exist_ok=True)

# --- Parámetros que puede cambiar -------------------------------------------
# Imágenes de ejemplo. Con None, el notebook elige la más representativa de cada
# clase (la más cercana a la mediana de color). Para fijarlas a mano, escriba el
# nombre del archivo, por ejemplo: IMG_MADURO = 'TestRipe(12).JPG'
IMG_MADURO = None
IMG_INMADURO = None
# Segundo del video del que se toman los tres fotogramas de la Figura 2.8.
SEGUNDO_VIDEO = 10.0
# -----------------------------------------------------------------------------

assert DIR_PALMA.exists(), f'No se encuentra el dataset de palma en: {DIR_PALMA}'
assert VIDEO_PIO.exists(), f'No se encuentra el video de pollos en: {VIDEO_PIO}'
print('Dataset de palma :', DIR_PALMA)
print('Video de pollos  :', VIDEO_PIO)
print('Salida           :', DIR_LIBRO)

In [ ]:
import numpy as np
import pandas as pd
import cv2
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import FancyBboxPatch, Rectangle
from matplotlib.ticker import FuncFormatter
from PIL import Image, ImageOps
from skimage.color import rgb2lab, lab2rgb, rgb2hsv

ANCHO_CM = 16.0                      # ancho de caja de texto del libro
ANCHO_IN = ANCHO_CM / 2.54
DPI = 300

# Tinta y colores de apoyo. Los datos categóricos no se distinguen solo por color:
# las cinco clases se muestran en paneles separados (Figura 2.7).
TINTA = '#0b0b0b'
TINTA_2 = '#52514e'
GRIS_FONDO = '#c9c8c2'
ACENTO = '#2a78d6'
REJILLA = '#e4e3de'

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 8,
    'axes.titlesize': 8,
    'axes.labelsize': 8,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'legend.fontsize': 7,
    'axes.edgecolor': TINTA_2,
    'axes.labelcolor': TINTA,
    'text.color': TINTA,
    'xtick.color': TINTA_2,
    'ytick.color': TINTA_2,
    'axes.linewidth': 0.6,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 110,
    'savefig.dpi': DPI,
    'figure.facecolor': 'white',
    'savefig.facecolor': 'white',
})

CLASES = ['unripe', 'ripe', 'overripe', 'damaged', 'empty']       # orden de presentación
NOMBRE_ES = {'unripe': 'Inmaduro', 'ripe': 'Maduro', 'overripe': 'Sobremaduro',
             'damaged': 'Dañado', 'empty': 'Vacío'}
EXT_IMG = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}

VALORES = {}          # cifras que se copian al manuscrito
FIGURAS = []          # rutas de las figuras generadas


def guardar(fig, ruta):
    # Guarda la figura a 300 dpi y la registra.
    fig.savefig(ruta, dpi=DPI, bbox_inches='tight', pad_inches=0.04)
    FIGURAS.append(str(ruta))
    print('Guardada:', ruta.name)


def leer_rgb(ruta, lado_max=None):
    # Lee una imagen como arreglo RGB uint8, respetando la orientación EXIF.
    # Con lado_max, la reduce para que su lado mayor no supere ese valor.
    with Image.open(ruta) as im:
        if lado_max is not None and im.format == 'JPEG':
            im.draft('RGB', (lado_max * 2, lado_max * 2))     # decodificación rápida
        im = ImageOps.exif_transpose(im).convert('RGB')
        if lado_max is not None and max(im.size) > lado_max:
            im.thumbnail((lado_max, lado_max), Image.LANCZOS)
        return np.asarray(im)


def redimensionar(img, lado_mayor):
    # Redimensiona conservando la proporción hasta que el lado mayor mida lado_mayor.
    h, w = img.shape[:2]
    escala = lado_mayor / max(h, w)
    nuevo = (max(1, round(w * escala)), max(1, round(h * escala)))
    interp = cv2.INTER_AREA if escala < 1 else cv2.INTER_CUBIC
    return cv2.resize(img, nuevo, interpolation=interp)


def recorte_cuadrado(img, lado=512):
    # Recorte cuadrado centrado, redimensionado a lado × lado píxeles.
    h, w = img.shape[:2]
    m = min(h, w)
    y0, x0 = (h - m) // 2, (w - m) // 2
    return cv2.resize(img[y0:y0 + m, x0:x0 + m], (lado, lado), interpolation=cv2.INTER_AREA)


def coma(x, dec=1):
    # Número con coma decimal, como se escribe en español.
    return f'{x:.{dec}f}'.replace('.', ',')


def eje_con_coma(eje, dec=2):
    eje.set_major_formatter(FuncFormatter(lambda v, _: coma(v, dec)))


def sin_ejes(ax):
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)


print('Bibliotecas listas. matplotlib', matplotlib.__version__, '| OpenCV', cv2.__version__)

## 2. Inventario del dataset de palma

Recorre las carpetas `train`, `val` y `test`, toma la clase del nombre de la carpeta y lee las dimensiones de cada imagen sin cargarla completa.

In [ ]:
filas = []
for ruta in sorted(DIR_PALMA.rglob('*')):
    if not ruta.is_file() or ruta.suffix.lower() not in EXT_IMG:
        continue
    partes = [p.lower() for p in ruta.relative_to(DIR_PALMA).parts[:-1]]
    clase = next((p for p in partes if p in CLASES), None)
    split = next((p for p in partes if p in ('train', 'val', 'test')), 'sin_split')
    if clase is None:
        continue
    with Image.open(ruta) as im:
        w, h = im.size
        orient = im.getexif().get(0x0112, 1)
    if orient in (5, 6, 7, 8):          # imagen girada 90° según EXIF
        w, h = h, w
    filas.append({'ruta': str(ruta), 'archivo': ruta.name, 'split': split, 'clase': clase,
                  'ancho': w, 'alto': h, 'bytes': ruta.stat().st_size})

inv = pd.DataFrame(filas)
assert len(inv) > 0, 'No se encontraron imágenes. Revise DIR_PALMA.'
faltan = [c for c in CLASES if c not in set(inv['clase'])]
assert not faltan, f'Faltan clases en el dataset: {faltan}'

conteo = inv.groupby(['clase', 'split']).size().unstack(fill_value=0).reindex(CLASES)
conteo['total'] = conteo.sum(axis=1)
print(conteo.rename(index=NOMBRE_ES).to_string())
print('\nTotal de imágenes:', len(inv))
if len(inv) != 452:
    print('AVISO: el manuscrito habla de 452 imágenes; aquí hay', len(inv))

inv['megapixeles'] = inv['ancho'] * inv['alto'] / 1e6
dims = (inv['ancho'].astype(str) + ' × ' + inv['alto'].astype(str)).value_counts()
med_w, med_h = int(inv['ancho'].median()), int(inv['alto'].median())
VALORES['palma_inventario'] = {
    'n_imagenes': int(len(inv)),
    'n_por_clase': {NOMBRE_ES[c]: int(conteo.loc[c, 'total']) for c in CLASES},
    'ancho_min': int(inv['ancho'].min()), 'ancho_mediana': med_w, 'ancho_max': int(inv['ancho'].max()),
    'alto_min': int(inv['alto'].min()), 'alto_mediana': med_h, 'alto_max': int(inv['alto'].max()),
    'dimensiones_distintas': int(len(dims)),
    'dimension_mas_frecuente': dims.index[0],
    'imagenes_con_dimension_mas_frecuente': int(dims.iloc[0]),
    'megapixeles_mediana': round(float(inv['megapixeles'].median()), 2),
    'tabla_2_1_fila_palma': {
        'dimensiones': f'{med_w} × {med_h}',
        'pixeles': int(med_w * med_h),
        'tamano_sin_comprimir_MB': round(med_w * med_h * 3 / 1e6, 2),
    },
    'tamano_archivo_mediana_MB': round(float(inv['bytes'].median()) / 1e6, 2),
}
print('\nDimensiones más frecuentes:')
print(dims.head(5).to_string())
print('\nFila de la Tabla 2.1 (fotografía original de palma, mediana):')
print(VALORES['palma_inventario']['tabla_2_1_fila_palma'])

## 3. Color por imagen y por clase (Tabla 2.3 y Figura 2.7)

Procedimiento, igual al descrito en la sección 2.4 del manuscrito:

1. redimensionar cada imagen a 256 píxeles en su lado mayor;
2. conservar la región central (mitad del ancho y mitad del alto);
3. convertir de sRGB a CIELAB con el iluminante D65 (`skimage.color.rgb2lab`) y promediar L\*, a\*, b\*;
4. agrupar por clase.

Además calcula una línea de base: asignar cada imagen de prueba a la clase cuyo color promedio (en entrenamiento) esté más cerca. Es el resultado que cualquier modelo debe superar.

In [ ]:
def color_central(ruta):
    img = leer_rgb(ruta, lado_max=256)
    h, w = img.shape[:2]
    centro = img[h // 4: h - h // 4, w // 4: w - w // 4].astype(np.float64) / 255.0
    lab = rgb2lab(centro).reshape(-1, 3)
    hsv = rgb2hsv(centro).reshape(-1, 3)
    ang = hsv[:, 0] * 2 * np.pi                                  # tono en radianes
    tono = (math.degrees(math.atan2(np.sin(ang).mean(), np.cos(ang).mean())) + 360) % 360
    return {'L': lab[:, 0].mean(), 'a': lab[:, 1].mean(), 'b': lab[:, 2].mean(),
            'tono_circular': tono, 'S': hsv[:, 1].mean(), 'V': hsv[:, 2].mean()}


registros = []
for i, fila in enumerate(inv.itertuples(index=False), start=1):
    registros.append(color_central(fila.ruta))
    if i % 50 == 0 or i == len(inv):
        print(f'  {i}/{len(inv)} imágenes procesadas')
color = pd.concat([inv.reset_index(drop=True), pd.DataFrame(registros)], axis=1)
color.drop(columns=['ruta']).to_csv(DIR_VAL / 'color_por_imagen.csv', index=False)

# --- Resumen por clase (Tabla 2.3) ---
res = color.groupby('clase')[['L', 'a', 'b', 'S', 'V']].agg(['mean', 'std']).reindex(CLASES)
tabla = pd.DataFrame({
    'Clase': [NOMBRE_ES[c] for c in CLASES],
    'Imágenes': [int((color['clase'] == c).sum()) for c in CLASES],
    'L*': [f"{res.loc[c, ('L', 'mean')]:.1f} ± {res.loc[c, ('L', 'std')]:.1f}" for c in CLASES],
    'a*': [f"{res.loc[c, ('a', 'mean')]:.1f} ± {res.loc[c, ('a', 'std')]:.1f}" for c in CLASES],
    'b*': [f"{res.loc[c, ('b', 'mean')]:.1f} ± {res.loc[c, ('b', 'std')]:.1f}" for c in CLASES],
})
tabla.to_csv(DIR_VAL / 'tabla_2_3_color_por_clase.csv', index=False)
print('\nTabla 2.3. Color promedio de la región central, por clase (media ± DE)')
print(tabla.to_string(index=False))

centros = color.groupby('clase')[['L', 'a', 'b']].mean().reindex(CLASES)

# --- Diferencia de color (Delta E 1976) entre las medias de clase ---
dE = pd.DataFrame(index=[NOMBRE_ES[c] for c in CLASES], columns=[NOMBRE_ES[c] for c in CLASES], dtype=float)
for c1 in CLASES:
    for c2 in CLASES:
        dE.loc[NOMBRE_ES[c1], NOMBRE_ES[c2]] = float(np.linalg.norm(centros.loc[c1] - centros.loc[c2]))
dE.round(1).to_csv(DIR_VAL / 'deltaE_entre_clases.csv')
print('\nDelta E entre las medias de clase:')
print(dE.round(1).to_string())

# --- Solapamiento de a* entre pares de clases vecinas ---
def solapamiento(c1, c2, col='a'):
    # Fracción de imágenes de ambas clases que cae en el intervalo que comparten
    # sus rangos entre los percentiles 5 y 95.
    x1 = color.loc[color['clase'] == c1, col]; x2 = color.loc[color['clase'] == c2, col]
    lo = max(x1.quantile(0.05), x2.quantile(0.05)); hi = min(x1.quantile(0.95), x2.quantile(0.95))
    if hi <= lo:
        return 0.0
    dentro = ((x1 >= lo) & (x1 <= hi)).sum() + ((x2 >= lo) & (x2 <= hi)).sum()
    return float(dentro / (len(x1) + len(x2)))

# --- Línea de base: clase del centroide de color más cercano ---
def f1_macro(y_real, y_pred, clases):
    f1s = []
    for c in clases:
        vp = int(((y_real == c) & (y_pred == c)).sum())
        fp = int(((y_real != c) & (y_pred == c)).sum())
        fn = int(((y_real == c) & (y_pred != c)).sum())
        f1s.append(0.0 if vp == 0 else 2 * vp / (2 * vp + fp + fn))
    return float(np.mean(f1s)), dict(zip(clases, f1s))

linea_base = None
if {'train', 'test'} <= set(color['split']):
    ent = color[color['split'] == 'train']; pru = color[color['split'] == 'test']
    cen = ent.groupby('clase')[['L', 'a', 'b']].mean().reindex(CLASES).dropna()
    X = pru[['L', 'a', 'b']].to_numpy()
    dist = np.linalg.norm(X[:, None, :] - cen.to_numpy()[None, :, :], axis=2)
    pred = np.array(cen.index)[dist.argmin(axis=1)]
    real = pru['clase'].to_numpy()
    f1m, f1c = f1_macro(real, pred, list(cen.index))
    linea_base = {
        'metodo': 'centroide de color CIELAB más cercano (entrenamiento → prueba)',
        'n_prueba': int(len(pru)),
        'exactitud': round(float((real == pred).mean()), 3),
        'f1_macro': round(f1m, 3),
        'f1_por_clase': {NOMBRE_ES[c]: round(v, 3) for c, v in f1c.items()},
        'referencia_modelo_YOLO11m_f1_macro_prueba': 0.846,
    }
    print('\nLínea de base por color (prueba):', json.dumps(linea_base, ensure_ascii=False, indent=1))

VALORES['color_por_clase'] = {
    'media_Lab': {NOMBRE_ES[c]: {k: round(float(v), 1) for k, v in centros.loc[c].items()} for c in CLASES},
    'de_Lab': {NOMBRE_ES[c]: {k: round(float(res.loc[c, (k, 'std')]), 1) for k in ('L', 'a', 'b')} for c in CLASES},
    'diferencia_a_maduro_menos_inmaduro': round(float(centros.loc['ripe', 'a'] - centros.loc['unripe', 'a']), 1),
    'deltaE_inmaduro_maduro': round(float(dE.loc['Inmaduro', 'Maduro']), 1),
    'deltaE_maduro_sobremaduro': round(float(dE.loc['Maduro', 'Sobremaduro']), 1),
    'solapamiento_a_inmaduro_maduro': round(solapamiento('unripe', 'ripe'), 2),
    'solapamiento_a_maduro_sobremaduro': round(solapamiento('ripe', 'overripe'), 2),
    'linea_base_color': linea_base,
}

## 4. Imágenes de ejemplo

Elige un racimo maduro y uno inmaduro para las figuras 2.1 a 2.6. Criterio: entre las fotos apaisadas de entrenamiento (en ellas el racimo suele llenar el encuadre), la más cercana a la mediana de color de su clase.

La celda guarda además `03_VALORES/candidatas_ejemplo.png` con las ocho mejores candidatas de cada clase y su nombre de archivo. Si prefiere otra, escriba el nombre en `IMG_MADURO` o `IMG_INMADURO` (celda de configuración) y vuelva a ejecutar desde aquí.

In [ ]:
def ordenadas(clase):
    # Imágenes de la clase ordenadas de más a menos representativa.
    sub = color[color['clase'] == clase]
    pref = sub[sub['split'] == 'train'] if (sub['split'] == 'train').any() else sub
    apais = pref[pref['ancho'] >= pref['alto']]
    if len(apais) >= 8:
        pref = apais
    med = pref[['L', 'a', 'b']].median().to_numpy()
    d = np.linalg.norm(pref[['L', 'a', 'b']].to_numpy() - med, axis=1)
    return pref.iloc[np.argsort(d)]


def representativa(clase, nombre_manual=None):
    if nombre_manual:
        sub = color[color['clase'] == clase]
        hit = sub[sub['archivo'] == nombre_manual]
        assert len(hit) > 0, f'No se encontró {nombre_manual} en la clase {clase}'
        return hit.iloc[0]
    return ordenadas(clase).iloc[0]


fila_mad = representativa('ripe', IMG_MADURO)
fila_inm = representativa('unripe', IMG_INMADURO)
print('Racimo maduro  :', fila_mad['archivo'], f"({fila_mad['ancho']} × {fila_mad['alto']})")
print('Racimo inmaduro:', fila_inm['archivo'], f"({fila_inm['ancho']} × {fila_inm['alto']})")

img_mad = leer_rgb(fila_mad['ruta'], lado_max=1536)
img_inm = leer_rgb(fila_inm['ruta'], lado_max=1536)
VALORES['imagenes_ejemplo'] = {
    'maduro': {'archivo': fila_mad['archivo'], 'ancho': int(fila_mad['ancho']), 'alto': int(fila_mad['alto'])},
    'inmaduro': {'archivo': fila_inm['archivo'], 'ancho': int(fila_inm['ancho']), 'alto': int(fila_inm['alto'])},
}

# Hoja de candidatas: la primera de cada fila es la elegida automáticamente.
N_CAND = 8
fig, axs = plt.subplots(2, N_CAND, figsize=(ANCHO_IN * 1.5, ANCHO_IN * 1.5 * 2 / N_CAND + 0.9), constrained_layout=True)
cand = {}
for f, (clase, elegida) in enumerate((('ripe', fila_mad), ('unripe', fila_inm))):
    lista = ordenadas(clase).head(N_CAND)
    cand[NOMBRE_ES[clase]] = list(lista['archivo'])
    for k in range(N_CAND):
        ax = axs[f, k]; sin_ejes(ax)
        if k < len(lista):
            fila = lista.iloc[k]
            ax.imshow(recorte_cuadrado(leer_rgb(fila['ruta'], lado_max=700), 300))
            ax.set_title(fila['archivo'], fontsize=4.2, color=TINTA,
                         fontweight='bold' if fila['archivo'] == elegida['archivo'] else 'normal')
    axs[f, 0].set_ylabel(NOMBRE_ES[clase], fontsize=8, color=TINTA)
fig.savefig(DIR_VAL / 'candidatas_ejemplo.png', dpi=200, bbox_inches='tight')
plt.show()
VALORES['candidatas_ejemplo'] = cand

## 4b. Figura 1.2 · Las cinco clases del conjunto de palma

Cinco ejemplos por clase, con rótulos en español. Se toman repartidos entre las imágenes más representativas de cada clase, para que la figura muestre casos típicos y algo de su variedad.

In [ ]:
N_EJ = 5
fig, axs = plt.subplots(len(CLASES), N_EJ, figsize=(ANCHO_IN, ANCHO_IN * len(CLASES) / N_EJ), constrained_layout=True)
mostradas = {}
for f, clase in enumerate(CLASES):
    orden = ordenadas(clase)
    tope = min(len(orden), max(N_EJ, int(len(orden) * 0.6)))
    pos = np.unique(np.linspace(0, tope - 1, min(N_EJ, tope)).round().astype(int))
    lista = orden.iloc[pos]
    mostradas[NOMBRE_ES[clase]] = list(lista['archivo'])
    for k in range(N_EJ):
        ax = axs[f, k]; sin_ejes(ax)
        if k < len(lista):
            ax.imshow(recorte_cuadrado(leer_rgb(lista.iloc[k]['ruta'], lado_max=900), 420))
    n_clase = int((color['clase'] == clase).sum())
    axs[f, 0].set_ylabel(f'{NOMBRE_ES[clase]}\nn = {n_clase}', fontsize=7.5, color=TINTA, linespacing=1.2)
guardar(fig, DIR_FIG1 / 'Fig_1_2_clases_palma.png')
plt.show()
VALORES['figura_1_2'] = {'ejemplos_por_clase': N_EJ, 'archivos': mostradas}

## 5. Figura 1.1 · Cadena del píxel a la decisión

Esquema con los seis eslabones y lo que significa cada uno en los dos casos del libro.

In [ ]:
from matplotlib.patches import Polygon

T1, T2, T3 = '#111111', '#4d4d4d', '#8a8a8a'
AZUL, AZUL_CLARO, GRIS_CLARO, LINEA = '#1f4e8c', '#e9eff8', '#f4f4f2', '#c9c9c4'

# Figura 1.1: la cadena del píxel a la decisión, con los dos casos del libro.
eslabones = ['Escena', 'Adquisición', 'Preproce-\nsamiento', 'Datos', 'Modelo', 'Indicador', 'Decisión']
preguntas = ['Lo que ocurre\nen el campo', '¿Qué registró\nla cámara?', '¿Está lista para\nanalizarse?',
             '¿Son correctos y\nrepresentativos?', '¿Qué hay en la\nimagen y dónde?',
             '¿Qué significa\npara el manejo?', '¿Qué se hace\ny cuándo?']
palma = ['Racimo de fruta\nfresca en la\nplantación', 'Fotografía en\nexteriores, con\nluz natural',
         'Redimensionar\na 384 píxeles', '452 imágenes\nen 5 clases',
         'Clasificador:\nuna clase\npor imagen', 'Estado del\nracimo y\nsu confianza',
         'Cosechar,\nesperar o\ndescartar']
pollos = ['Lote de aves\nen el galpón', 'Video cenital,\n1920 × 1080\na 30 fps',
          'Redimensionar\na 960 píxeles', '1 487 imágenes,\n327 288 aves\nanotadas',
          'Detector y\nseguimiento: una\ncaja por ave', 'Conteo,\nocupación por\nzona, actividad',
          'Ajustar el\nambiente o\nrevisar zonas']
caps = ['1', '2 y 3', '3 y 4', '5', '6 a 9', '9 y 11', '10 y 11']

W, H = ANCHO_CM, 7.75
x0 = 1.95                         # ancho de la columna de rótulos
cw = (W - x0) / 7                 # ancho de cada columna
fig = plt.figure(figsize=(W / 2.54, H / 2.54))
ax = fig.add_axes([0, 0, 1, 1]); ax.set_xlim(0, W); ax.set_ylim(0, H); ax.axis('off')

def cx(i):
    return x0 + cw * (i + 0.5)

# --- llaves superiores: quién hace cada parte ---
y_ll = H - 0.34
for i0, i1, texto in ((1, 5, 'SISTEMA DE VISIÓN ARTIFICIAL'), (6, 6, 'PERSONA')):
    xa, xb = x0 + cw * i0 + 0.10, x0 + cw * (i1 + 1) - 0.10
    ax.plot([xa, xa, xb, xb], [y_ll - 0.16, y_ll, y_ll, y_ll - 0.16], color=T3, lw=0.6,
            solid_capstyle='butt')
    ax.text((xa + xb) / 2, y_ll + 0.07, texto, ha='center', va='bottom', fontsize=4.9,
            color=T2, fontweight='bold')

# --- fila de eslabones: origen + seis flechas encadenadas ---
yt, hh = H - 0.72, 1.02           # borde superior y alto
yb = yt - hh; ym = yt - hh / 2; p = 0.24   # punta de la flecha
ax.add_patch(FancyBboxPatch((x0 + 0.10, yb), cw - 0.34, hh, boxstyle='round,pad=0,rounding_size=0.12',
                            facecolor='white', edgecolor=T2, lw=0.8))
ax.text(x0 + 0.10 + (cw - 0.34) / 2, ym, eslabones[0], ha='center', va='center', fontsize=6.6,
        fontweight='bold', color=T1)
for i in range(1, 7):
    xa = x0 + cw * i - 0.12; xb = x0 + cw * (i + 1) - 0.12
    if i == 6:
        xb = W - 0.30
    relleno = T1 if i == 6 else AZUL
    ax.add_patch(Polygon([(xa, yt), (xb - p, yt), (xb, ym), (xb - p, yb), (xa, yb), (xa + p, ym)],
                         closed=True, facecolor=relleno, edgecolor='white', lw=1.2, joinstyle='miter'))
    ax.text(xa + p + 0.10, yt - 0.17, str(i), ha='left', va='center', fontsize=4.6,
            color='white', alpha=0.9)
    ax.text((xa + xb) / 2 + 0.06, ym - 0.04, eslabones[i], ha='center', va='center', fontsize=5.9,
            fontweight='bold', color='white', linespacing=0.95)

# --- filas de contenido ---
filas = [('PREGUNTA QUE\nRESPONDE', preguntas, 1.00, None, T2, 'italic'),
         ('CASO PALMA\nAFRICANA', palma, 1.38, AZUL_CLARO, T1, 'normal'),
         ('CASO POLLOS\nDE ENGORDE', pollos, 1.38, GRIS_CLARO, T1, 'normal'),
         ('CAPÍTULOS', caps, 0.58, None, T2, 'normal')]
y = yb - 0.20
for rotulo, celdas, alto, fondo, tinta_fila, estilo in filas:
    if fondo:
        ax.add_patch(Rectangle((0.12, y - alto), W - 0.24, alto, facecolor=fondo, edgecolor='none'))
    else:
        ax.plot([0.12, W - 0.12], [y - alto, y - alto], color=LINEA, lw=0.5)
    ax.text(0.26, y - alto / 2, rotulo, ha='left', va='center', fontsize=5.0, fontweight='bold',
            color=T2, linespacing=1.15)
    for i, t in enumerate(celdas):
        ax.text(cx(i), y - alto / 2, t, ha='center', va='center', fontsize=5.7, color=tinta_fila,
                style=estilo, linespacing=1.18)
    y -= alto + (0.10 if fondo else 0.0)
# líneas verticales tenues entre columnas, solo en las filas de casos
y_sup = yb - 0.20 - 1.00; y_inf = y_sup - 2 * 1.38 - 0.10
for i in range(1, 7):
    ax.plot([x0 + cw * i - 0.03] * 2, [y_inf + 0.12, y_sup - 0.12], color='white', lw=1.4)

# --- banda inferior: propagación del error ---
yf = 0.50
ax.annotate('', xy=(W - 0.30, yf), xytext=(x0 + 0.10, yf),
            arrowprops=dict(arrowstyle='-|>', color=T3, lw=0.8, mutation_scale=8))
ax.text((x0 + W) / 2, yf - 0.13, 'Los errores se propagan hacia adelante: el resultado final nunca es mejor que el eslabón más débil',
        ha='center', va='top', fontsize=5.6, style='italic', color=T2)
ruta_11 = DIR_FIG1 / 'Fig_1_1_cadena_pixel_decision.png'
fig.savefig(ruta_11, dpi=DPI, facecolor='white')
FIGURAS.append(str(ruta_11)); print('Guardada:', ruta_11.name)
plt.show()

## 6. Figura 2.1 · La imagen a tres escalas

In [ ]:
base = redimensionar(img_mad, 768)                # imagen de trabajo: 768 px de lado mayor
H, W = base.shape[:2]
cy, cx = H // 2, W // 2
r48 = base[cy - 24: cy + 24, cx - 24: cx + 24]
r8 = base[cy - 4: cy + 4, cx - 4: cx + 4]

fig, axs = plt.subplots(1, 3, figsize=(ANCHO_IN, ANCHO_IN * 0.40), constrained_layout=True,
                        gridspec_kw={'width_ratios': [W / H, 1, 1]})
axs[0].imshow(base)
axs[0].add_patch(Rectangle((cx - 24, cy - 24), 48, 48, fill=False, edgecolor='white', linewidth=1.6))
axs[0].add_patch(Rectangle((cx - 24, cy - 24), 48, 48, fill=False, edgecolor=TINTA, linewidth=0.7))
axs[0].set_title(f'(a) Imagen completa\n{W} × {H} píxeles', linespacing=1.1)

axs[1].imshow(r48, interpolation='nearest')
axs[1].add_patch(Rectangle((24 - 4 - 0.5, 24 - 4 - 0.5), 8, 8, fill=False, edgecolor='white', linewidth=1.6))
axs[1].add_patch(Rectangle((24 - 4 - 0.5, 24 - 4 - 0.5), 8, 8, fill=False, edgecolor=TINTA, linewidth=0.7))
axs[1].set_title('(b) Recorte\n48 × 48 píxeles', linespacing=1.1)

axs[2].imshow(r8, interpolation='nearest')
for i in range(8):
    for j in range(8):
        r, g, b = (int(v) for v in r8[i, j])
        lum = 0.299 * r + 0.587 * g + 0.114 * b
        axs[2].text(j, i, f'{r}\n{g}\n{b}', ha='center', va='center', fontsize=3.8,
                    color='white' if lum < 140 else 'black', linespacing=0.95)
axs[2].set_xticks(np.arange(-0.5, 8, 1), minor=True); axs[2].set_yticks(np.arange(-0.5, 8, 1), minor=True)
axs[2].grid(which='minor', color='white', linewidth=0.4)
axs[2].tick_params(which='minor', length=0)
axs[2].set_title('(c) Recorte de 8 × 8 píxeles\nvalores R, G, B', linespacing=1.1)
for ax in axs:
    sin_ejes(ax)
guardar(fig, DIR_FIG2 / 'Fig_2_1_pixel_ampliacion.png')
plt.show()
VALORES['figura_2_1'] = {'imagen': fila_mad['archivo'], 'dimensiones_mostradas': f'{W} × {H}',
                         'nota': 'La imagen se redimensionó a 768 px de lado mayor antes de recortar.'}

## 7. Figura 2.2 · Una misma fotografía a cuatro resoluciones

In [ ]:
lados = [None, 384, 96, 24]
fig, axs = plt.subplots(1, 4, figsize=(ANCHO_IN, ANCHO_IN * 0.30), constrained_layout=True)
detalle = {}
for ax, lado, letra in zip(axs, lados, 'abcd'):
    if lado is None:
        v = img_mad
        titulo = f"({letra}) Original: {fila_mad['ancho']} × {fila_mad['alto']}"
    else:
        v = redimensionar(img_mad, lado)
        titulo = f'({letra}) {v.shape[1]} × {v.shape[0]}'
        detalle[str(lado)] = f'{v.shape[1]} × {v.shape[0]}'
    ax.imshow(v, interpolation='nearest')
    ax.set_title(titulo, linespacing=1.1); sin_ejes(ax)
guardar(fig, DIR_FIG2 / 'Fig_2_2_resolucion_espacial.png')
plt.show()
VALORES['figura_2_2'] = {'imagen': fila_mad['archivo'], 'dimensiones': detalle}

## 8. Figuras 2.3, 2.4 y 2.5 · Canales en RGB, HSV y CIELAB

In [ ]:
# Las figuras 2.3 a 2.6 usan el recorte cuadrado central de cada imagen de ejemplo.
ej = [('Maduro', recorte_cuadrado(img_mad)), ('Inmaduro', recorte_cuadrado(img_inm))]

# Mapas divergentes con punto medio gris neutro para a* y b*.
MAPA_A = LinearSegmentedColormap.from_list('a_estrella', ['#00823c', '#d9d9d9', '#c8281e'])
MAPA_B = LinearSegmentedColormap.from_list('b_estrella', ['#1f5fbf', '#d9d9d9', '#d9a400'])


def figura_canales(nombre, columnas):
    # columnas: lista de (título, función(img)->2D, cmap, vmin, vmax, etiqueta de barra)
    fig = plt.figure(figsize=(ANCHO_IN, ANCHO_IN * 0.5 + 0.62), constrained_layout=True)
    gs = fig.add_gridspec(3, 4, height_ratios=[1, 1, 0.055])
    axs = np.array([[fig.add_subplot(gs[f, c]) for c in range(4)] for f in range(2)])
    for fila, (etq, im) in enumerate(ej):
        axs[fila, 0].imshow(im)
        axs[fila, 0].set_ylabel(etq, fontsize=8, color=TINTA)
        if fila == 0:
            axs[fila, 0].set_title('Imagen')
        for col, (titulo, fn, cmap, vmin, vmax, barra) in enumerate(columnas, start=1):
            m = axs[fila, col].imshow(fn(im), cmap=cmap, vmin=vmin, vmax=vmax)
            if fila == 0:
                axs[fila, col].set_title(titulo)
            else:
                cb = fig.colorbar(m, cax=fig.add_subplot(gs[2, col]), orientation='horizontal')
                cb.set_label(barra, fontsize=6.5); cb.ax.tick_params(labelsize=6, length=2)
                cb.outline.set_linewidth(0.4)
                if vmax <= 1:
                    eje_con_coma(cb.ax.xaxis, 1)
    for ax in axs.ravel():
        ax.set_xticks([]); ax.set_yticks([])
        for sp in ax.spines.values():
            sp.set_visible(False)
    guardar(fig, DIR_FIG2 / nombre)
    plt.show()


figura_canales('Fig_2_3_canales_RGB.png', [
    ('Canal rojo (R)', lambda im: im[:, :, 0], 'gray', 0, 255, 'Nivel (0–255)'),
    ('Canal verde (G)', lambda im: im[:, :, 1], 'gray', 0, 255, 'Nivel (0–255)'),
    ('Canal azul (B)', lambda im: im[:, :, 2], 'gray', 0, 255, 'Nivel (0–255)'),
])

figura_canales('Fig_2_4_canales_HSV.png', [
    ('Tono (H)', lambda im: rgb2hsv(im)[:, :, 0] * 360, 'hsv', 0, 360, 'Grados'),
    ('Saturación (S)', lambda im: rgb2hsv(im)[:, :, 1], 'gray', 0, 1, 'Saturación (0–1)'),
    ('Valor (V)', lambda im: rgb2hsv(im)[:, :, 2], 'gray', 0, 1, 'Valor (0–1)'),
])

figura_canales('Fig_2_5_canales_CIELAB.png', [
    ('Luminosidad (L*)', lambda im: rgb2lab(im)[:, :, 0], 'gray', 0, 100, 'L*'),
    ('Verde – rojo (a*)', lambda im: rgb2lab(im)[:, :, 1], MAPA_A, -60, 60, 'a*'),
    ('Azul – amarillo (b*)', lambda im: rgb2lab(im)[:, :, 2], MAPA_B, -60, 60, 'b*'),
])

# Correlación entre canales RGB (apoya el texto de la sección 2.2).
corr = {}
for etq, im in ej:
    c = np.corrcoef(im.reshape(-1, 3).T.astype(np.float64))
    corr[etq] = {'R-G': round(float(c[0, 1]), 2), 'R-B': round(float(c[0, 2]), 2), 'G-B': round(float(c[1, 2]), 2)}
VALORES['correlacion_canales_RGB'] = corr
print('Correlación entre canales RGB:', corr)

## 9. Figura 2.6 · Histogramas

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(ANCHO_IN, ANCHO_IN * 0.52), constrained_layout=True,
                        gridspec_kw={'width_ratios': [0.75, 1.3, 1.3]})
niveles = np.arange(256)
a_todas = np.concatenate([rgb2lab(im)[:, :, 1].ravel() for _, im in ej])
a_min = math.floor(np.percentile(a_todas, 0.1) / 10) * 10
a_max = math.ceil(np.percentile(a_todas, 99.9) / 10) * 10
canales = [('R', '#c8281e', '-'), ('G', '#00823c', '--'), ('B', '#1f5fbf', ':')]
hist_info = {}
tope_rgb = 0
for fila, (etq, im) in enumerate(ej):
    axs[fila, 0].imshow(im); sin_ejes(axs[fila, 0]); axs[fila, 0].set_ylabel(etq, fontsize=8, color=TINTA)
    info = {}
    for k, (nombre, col, estilo) in enumerate(canales):
        h = np.bincount(im[:, :, k].ravel(), minlength=256) / im[:, :, k].size
        axs[fila, 1].plot(niveles, h, color=col, linestyle=estilo, linewidth=1.1, label=nombre)
        tope_rgb = max(tope_rgb, float(np.percentile(h, 99.5)))
        info[nombre] = {'media': round(float(im[:, :, k].mean()), 1), 'de': round(float(im[:, :, k].std()), 1),
                        'pct_en_0': round(float((im[:, :, k] == 0).mean() * 100), 2),
                        'pct_en_255': round(float((im[:, :, k] == 255).mean() * 100), 2)}
    a = rgb2lab(im)[:, :, 1].ravel()
    axs[fila, 2].hist(a, bins=np.arange(a_min, a_max + 2, 2), density=True, color=ACENTO, edgecolor='white', linewidth=0.3)
    axs[fila, 2].axvline(a.mean(), color=TINTA, linewidth=0.8)
    axs[fila, 2].text(0.98, 0.95, f'media = {coma(a.mean())}', transform=axs[fila, 2].transAxes,
                      ha='right', va='top', fontsize=6.5, color=TINTA,
                      bbox=dict(facecolor='white', edgecolor='none', pad=1.5, alpha=0.9))
    info['a_estrella'] = {'media': round(float(a.mean()), 1), 'de': round(float(a.std()), 1)}
    hist_info[etq] = info
for fila in range(2):
    axs[fila, 1].set_xlim(0, 255); axs[fila, 1].set_ylim(0, tope_rgb * 1.15)
    axs[fila, 1].set_ylabel('Proporción de píxeles'); axs[fila, 1].grid(axis='y', color=REJILLA, linewidth=0.5)
    axs[fila, 2].set_xlim(a_min, a_max); axs[fila, 2].set_ylabel('Densidad')
    axs[fila, 2].grid(axis='y', color=REJILLA, linewidth=0.5)
    for ax in (axs[fila, 1], axs[fila, 2]):
        ax.set_axisbelow(True)
    eje_con_coma(axs[fila, 1].yaxis, 3)
    eje_con_coma(axs[fila, 2].yaxis, 3)
tope_a = max(axs[0, 2].get_ylim()[1], axs[1, 2].get_ylim()[1])
for fila in range(2):
    axs[fila, 2].set_ylim(0, tope_a)
axs[0, 0].set_title('Imagen'); axs[0, 1].set_title('Histograma por canal'); axs[0, 2].set_title('Histograma de a*')
axs[1, 1].set_xlabel('Nivel de intensidad (0–255)'); axs[1, 2].set_xlabel('a* (verde ← 0 → rojo)')
axs[1, 1].legend(frameon=False, ncol=3, loc='upper center', bbox_to_anchor=(0.5, -0.30), handlelength=2.6)
guardar(fig, DIR_FIG2 / 'Fig_2_6_histogramas.png')
plt.show()
VALORES['histogramas'] = hist_info
print(json.dumps(hist_info, ensure_ascii=False, indent=1))

## 10. Figura 2.7 · Color de las cinco clases

Tres paneles: (a) color promedio de cada clase; (b) posición de cada imagen en el plano a\*–b\*, con un panel por clase y el resto de las imágenes en gris; (c) distribución de a\* por clase.

In [ ]:
fig = plt.figure(figsize=(ANCHO_IN, ANCHO_IN * 1.0), constrained_layout=True)
sub = fig.subfigures(3, 1, height_ratios=[0.50, 1.0, 0.95], hspace=0.04)
TIT = dict(x=0.01, ha='left', fontsize=8, fontweight='bold')

# (a) Color promedio de cada clase
sub[0].suptitle('(a) Color promedio de la región central', **TIT)
axs_a = sub[0].subplots(1, 5)
for ax, c in zip(axs_a, CLASES):
    rgb = lab2rgb(centros.loc[c].to_numpy().reshape(1, 1, 3)).reshape(3)
    ax.imshow(np.ones((9, 16, 3)) * rgb)
    ax.set_title(f"{NOMBRE_ES[c]}\nn = {int((color['clase'] == c).sum())}", fontsize=7, linespacing=1.15)
    sin_ejes(ax)

# (b) Plano a*–b*: un panel por clase, las demás en gris
sub[1].suptitle('(b) Cada imagen en el plano a*–b* (en gris, las demás clases)', **TIT)
axs_b = sub[1].subplots(1, 5, sharex=True, sharey=True)
for k, (ax, c) in enumerate(zip(axs_b, CLASES)):
    otros = color[color['clase'] != c]; esta = color[color['clase'] == c]
    ax.scatter(otros['a'], otros['b'], s=5, color=GRIS_FONDO, linewidths=0)
    ax.scatter(esta['a'], esta['b'], s=10, color=ACENTO, edgecolors='white', linewidths=0.3)
    ax.set_title(NOMBRE_ES[c], fontsize=7); ax.set_xlabel('a*')
    ax.grid(color=REJILLA, linewidth=0.5); ax.set_axisbelow(True)
    if k == 0:
        ax.set_ylabel('b*')
axs_b[0].set_xlim(color['a'].min() - 3, color['a'].max() + 3)
axs_b[0].set_ylim(color['b'].min() - 3, color['b'].max() + 3)

# (c) Distribución de a* por clase
sub[2].suptitle('(c) Distribución de a* por clase (caja: cuartiles; puntos: imágenes)', **TIT)
ax = sub[2].subplots(1, 1)
datos = [color.loc[color['clase'] == c, 'a'].to_numpy() for c in CLASES]
ax.boxplot(datos, vert=False, widths=0.55, showfliers=False, patch_artist=True,
           boxprops=dict(facecolor='#e6effb', edgecolor=ACENTO, linewidth=0.8),
           medianprops=dict(color=TINTA, linewidth=1.2),
           whiskerprops=dict(color=ACENTO, linewidth=0.8), capprops=dict(color=ACENTO, linewidth=0.8))
rng = np.random.default_rng(2026)
for k, d in enumerate(datos, start=1):
    ax.scatter(d, k + rng.uniform(-0.2, 0.2, size=len(d)), s=5, color=TINTA_2, alpha=0.5,
               linewidths=0, zorder=3)
ax.set_yticks(range(1, len(CLASES) + 1))
ax.set_yticklabels([NOMBRE_ES[c] for c in CLASES])
ax.invert_yaxis()
ax.set_xlabel('a* promedio de la región central (verde ← 0 → rojo)')
ax.grid(axis='x', color=REJILLA, linewidth=0.5); ax.set_axisbelow(True)
guardar(fig, DIR_FIG2 / 'Fig_2_7_color_por_clase.png')
plt.show()

## 11. Figura 2.8 · Tres fotogramas consecutivos y su diferencia

Lee tres fotogramas seguidos del video a partir de `SEGUNDO_VIDEO` y calcula la diferencia absoluta entre el primero y el tercero.

In [ ]:
cap = cv2.VideoCapture(str(VIDEO_PIO))
assert cap.isOpened(), f'No se pudo abrir el video: {VIDEO_PIO}'
fps = float(cap.get(cv2.CAP_PROP_FPS))
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
vw = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); vh = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
duracion = n_frames / fps if fps > 0 else float('nan')

inicio = int(round(SEGUNDO_VIDEO * fps))
inicio = max(0, min(inicio, n_frames - 3))
cap.set(cv2.CAP_PROP_POS_FRAMES, inicio)
frames = []
for _ in range(3):
    ok, f = cap.read()
    assert ok, 'No se pudieron leer tres fotogramas consecutivos; cambie SEGUNDO_VIDEO.'
    frames.append(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))

# Actividad media: diferencia entre fotogramas consecutivos durante 2 segundos.
cap.set(cv2.CAP_PROP_POS_FRAMES, inicio)
ok, previo = cap.read(); previo = cv2.cvtColor(previo, cv2.COLOR_BGR2GRAY).astype(np.int16)
difs = []
for _ in range(int(min(2 * fps, n_frames - inicio - 1))):
    ok, f = cap.read()
    if not ok:
        break
    g = cv2.cvtColor(f, cv2.COLOR_BGR2GRAY).astype(np.int16)
    difs.append(float(np.abs(g - previo).mean())); previo = g
cap.release()

gris = [cv2.cvtColor(f, cv2.COLOR_RGB2GRAY).astype(np.int16) for f in frames]
dif13 = np.abs(gris[2] - gris[0]).astype(np.uint8)
umbral = 25
pct_mov = float((dif13 > umbral).mean() * 100)

fig, axs = plt.subplots(2, 2, figsize=(ANCHO_IN, ANCHO_IN * (vh / vw) + 0.45), constrained_layout=True)
for k in range(3):
    ax = axs.ravel()[k]
    ax.imshow(frames[k])
    ax.set_title(f'({"abc"[k]}) Fotograma {inicio + k}  ·  t = {coma((inicio + k) / fps, 3)} s')
axs[1, 1].imshow(dif13, cmap='gray', vmin=0, vmax=max(60, int(np.percentile(dif13, 99.9))))
axs[1, 1].set_title('(d) Diferencia absoluta entre (a) y (c): claro = cambio')
for ax in axs.ravel():
    sin_ejes(ax)
guardar(fig, DIR_FIG2 / 'Fig_2_8_fotogramas_consecutivos.png')
plt.show()

bytes_archivo = VIDEO_PIO.stat().st_size
bytes_crudo = n_frames * vw * vh * 3
VALORES['video_pio'] = {
    'archivo': VIDEO_PIO.name,
    'ancho': vw, 'alto': vh, 'fps': round(fps, 3), 'fotogramas': n_frames,
    'duracion_s': round(duracion, 2),
    'tamano_archivo_MB': round(bytes_archivo / 1e6, 1),
    'tamano_sin_comprimir_GB': round(bytes_crudo / 1e9, 2),
    'razon_compresion': round(bytes_crudo / bytes_archivo, 0),
    'conserva_resolucion_nativa_1920x1080': bool(vw == 1920 and vh == 1080),
    'fotogramas_figura_2_8': [inicio, inicio + 1, inicio + 2],
    'diferencia_media_entre_consecutivos_niveles': round(float(np.mean(difs)), 2) if difs else None,
    'pct_pixeles_con_cambio_mayor_a_25_entre_a_y_c': round(pct_mov, 2),
}
print(json.dumps(VALORES['video_pio'], ensure_ascii=False, indent=1))

## 12. Valores para el manuscrito

Guarda `valores_para_texto.json`. Con ese archivo se completan las cifras marcadas con ⟦ ⟧ en los capítulos 1 y 2.

In [ ]:
VALORES['figuras_generadas'] = [Path(f).name for f in FIGURAS]
ruta_json = DIR_VAL / 'valores_para_texto.json'
with open(ruta_json, 'w', encoding='utf-8') as fh:
    json.dump(VALORES, fh, ensure_ascii=False, indent=2)

print('Figuras generadas:', len(FIGURAS))
for f in FIGURAS:
    print('  ', Path(f).relative_to(DIR_LIBRO))
print('\nValores guardados en:', ruta_json.relative_to(DIR_LIBRO))
print('\nResumen para el texto')
print('  Tabla 2.1, fila de palma :', VALORES['palma_inventario']['tabla_2_1_fila_palma'])
print('  a* maduro − inmaduro     :', VALORES['color_por_clase']['diferencia_a_maduro_menos_inmaduro'])
print('  ΔE inmaduro–maduro       :', VALORES['color_por_clase']['deltaE_inmaduro_maduro'])
print('  ΔE maduro–sobremaduro    :', VALORES['color_por_clase']['deltaE_maduro_sobremaduro'])
lb = VALORES['color_por_clase']['linea_base_color']
if lb:
    print('  Línea de base por color  : exactitud', lb['exactitud'], '| F1 macro', lb['f1_macro'])
v = VALORES['video_pio']
print(f"  Video                    : {v['ancho']} × {v['alto']}, {v['fps']} fps, {v['fotogramas']} fotogramas, "
      f"compresión {int(v['razon_compresion'])}:1")